# 🧠 Real Neural Network for Gender Classification

This notebook creates a **real neural network** that your React app can connect to for actual inference results.

## 🚀 Quick Start
1. **Run all cells** in order (Runtime → Run all)
2. **Copy the ngrok URL** from the last cell
3. **Paste it in your React app** to connect

---

In [ ]:
# 📦 Install required packages
!pip install flask flask-cors tensorflow pandas numpy scikit-learn pyngrok -q
print("✅ All packages installed successfully!")

In [ ]:
# 📚 Import libraries
import tensorflow as tf
from tensorflow.keras import layers, Model, optimizers, losses
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from flask import Flask, request, jsonify
from flask_cors import CORS
import json
import threading
import time
from pyngrok import ngrok

print(f"✅ TensorFlow version: {tf.__version__}")
print("✅ All libraries imported successfully!")

In [ ]:
# 📊 Create sample dataset for gender classification
sample_names = [
    # Female names
    ('Sarah', 'Female'), ('Emma', 'Female'), ('Jessica', 'Female'), 
    ('Ashley', 'Female'), ('Amanda', 'Female'), ('Michelle', 'Female'),
    ('Lisa', 'Female'), ('Emily', 'Female'), ('Kimberly', 'Female'),
    ('Jennifer', 'Female'), ('Nicole', 'Female'), ('Elizabeth', 'Female'),
    ('Rebecca', 'Female'), ('Maria', 'Female'), ('Stephanie', 'Female'),
    ('Rachel', 'Female'), ('Catherine', 'Female'), ('Angela', 'Female'),
    ('Samantha', 'Female'), ('Katherine', 'Female'), ('Christina', 'Female'),
    ('Linda', 'Female'), ('Barbara', 'Female'), ('Susan', 'Female'),
    ('Karen', 'Female'), ('Nancy', 'Female'), ('Donna', 'Female'),
    ('Carol', 'Female'), ('Ruth', 'Female'), ('Sharon', 'Female'),
    
    # Male names
    ('Michael', 'Male'), ('David', 'Male'), ('Robert', 'Male'),
    ('William', 'Male'), ('Christopher', 'Male'), ('Matthew', 'Male'),
    ('Joshua', 'Male'), ('Andrew', 'Male'), ('Daniel', 'Male'),
    ('James', 'Male'), ('John', 'Male'), ('Ryan', 'Male'),
    ('Nicholas', 'Male'), ('Alexander', 'Male'), ('Jonathan', 'Male'),
    ('Tyler', 'Male'), ('Brandon', 'Male'), ('Anthony', 'Male'),
    ('Steven', 'Male'), ('Thomas', 'Male'), ('Kevin', 'Male'),
    ('Paul', 'Male'), ('Mark', 'Male'), ('Donald', 'Male'),
    ('Kenneth', 'Male'), ('Richard', 'Male'), ('Charles', 'Male'),
    ('Joseph', 'Male'), ('Edward', 'Male'), ('George', 'Male')
]

print(f"📊 Dataset created with {len(sample_names)} names")
female_count = sum(1 for _, gender in sample_names if gender == 'Female')
male_count = sum(1 for _, gender in sample_names if gender == 'Male')
print(f"   👩 Female: {female_count} names")
print(f"   👨 Male: {male_count} names")

In [ ]:
# 🤖 Gender Classification Model Class
class GenderClassificationModel:
    def __init__(self):
        self.model = None
        self.char_to_idx = None
        self.idx_to_char = None
        self.label_encoder = None
        self.max_length = 20
        self.training_history = None
        
    def prepare_data(self, names_data):
        """Prepare and preprocess the data"""
        df = pd.DataFrame(names_data, columns=['name', 'gender'])
        
        # Encode labels
        self.label_encoder = LabelEncoder()
        y = self.label_encoder.fit_transform(df['gender'])
        
        # Create character-level tokenizer
        all_chars = set(''.join(df['name'].str.lower()))
        self.char_to_idx = {char: i+1 for i, char in enumerate(sorted(all_chars))}
        self.char_to_idx['<PAD>'] = 0
        self.idx_to_char = {v: k for k, v in self.char_to_idx.items()}
        
        # Convert names to sequences
        X = []
        for name in df['name']:
            sequence = [self.char_to_idx.get(c.lower(), 0) for c in name]
            sequence = sequence[:self.max_length]  # Truncate
            sequence += [0] * (self.max_length - len(sequence))  # Pad
            X.append(sequence)
        
        return np.array(X), y
    
    def build_model(self):
        """Build the neural network model"""
        # Input layer
        input_layer = layers.Input(shape=(self.max_length,), name='name_input')
        
        # Embedding layer
        embedding = layers.Embedding(
            input_dim=len(self.char_to_idx),
            output_dim=64,
            input_length=self.max_length,
            name='embedding'
        )(input_layer)
        
        # LSTM layer
        lstm = layers.LSTM(128, name='lstm', return_sequences=False)(embedding)
        
        # Dense layers
        dense1 = layers.Dense(64, activation='relu', name='dense1')(lstm)
        dropout = layers.Dropout(0.3, name='dropout')(dense1)
        dense2 = layers.Dense(32, activation='relu', name='dense2')(dropout)
        
        # Output layer
        output = layers.Dense(1, activation='sigmoid', name='output')(dense2)
        
        # Create model
        model = Model(inputs=input_layer, outputs=output, name='gender_classifier')
        
        # Compile model
        model.compile(
            optimizer=optimizers.Adam(learning_rate=0.001),
            loss=losses.BinaryCrossentropy(),
            metrics=['accuracy']
        )
        
        return model
    
    def train(self, names_data):
        """Train the model"""
        X, y = self.prepare_data(names_data)
        
        # Split data
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )
        
        # Build model
        self.model = self.build_model()
        
        print("🏗️ Model Architecture:")
        self.model.summary()
        
        # Train model
        print("\n🚀 Starting training...")
        history = self.model.fit(
            X_train, y_train,
            validation_data=(X_test, y_test),
            epochs=50,
            batch_size=16,
            verbose=1
        )
        
        self.training_history = history.history
        
        # Evaluate
        test_loss, test_accuracy = self.model.evaluate(X_test, y_test, verbose=0)
        print(f"\n✅ Training completed!")
        print(f"📊 Test Accuracy: {test_accuracy:.4f}")
        print(f"📉 Test Loss: {test_loss:.4f}")
        
        return history
    
    def predict(self, name):
        """Make a prediction for a single name"""
        if self.model is None:
            raise ValueError("Model not trained yet!")
        
        # Preprocess the name
        sequence = [self.char_to_idx.get(c.lower(), 0) for c in name]
        sequence = sequence[:self.max_length]
        sequence += [0] * (self.max_length - len(sequence))
        
        # Make prediction
        input_data = np.array([sequence])
        prediction = self.model.predict(input_data, verbose=0)[0][0]
        
        # Get layer activations
        layer_outputs = self.get_layer_activations(input_data)
        
        return {
            'prediction': float(prediction),
            'gender': 'Female' if prediction > 0.5 else 'Male',
            'confidence': float(abs(prediction - 0.5) * 2),
            'layer_activations': layer_outputs
        }
    
    def get_layer_activations(self, input_data):
        """Get activations from each layer"""
        if self.model is None:
            return {}
        
        layer_outputs = {}
        
        # Create models for each layer to get intermediate outputs
        for i, layer in enumerate(self.model.layers[1:], 1):  # Skip input layer
            if hasattr(layer, 'output'):
                temp_model = Model(inputs=self.model.input, outputs=layer.output)
                output = temp_model.predict(input_data, verbose=0)
                
                # Calculate activation intensity
                if len(output.shape) > 2:  # For LSTM/embedding layers
                    activation_intensity = np.mean(np.abs(output))
                else:  # For dense layers
                    activation_intensity = np.mean(np.abs(output[0]))
                
                layer_outputs[layer.name] = float(activation_intensity)
        
        return layer_outputs

print("✅ GenderClassificationModel class defined!")

In [ ]:
# 🎯 Train the model
print("🚀 Initializing and training the gender classification model...")
gender_model = GenderClassificationModel()
history = gender_model.train(sample_names)
print("\n🎉 Model training completed successfully!")

In [ ]:
# 🧪 Test the model with sample predictions
test_names = ['Sarah', 'Michael', 'Emma', 'David', 'Jessica', 'Robert']
print("🧪 Testing the trained model:")
print("-" * 50)

for name in test_names:
    result = gender_model.predict(name)
    confidence_emoji = "🔥" if result['confidence'] > 0.8 else "✅" if result['confidence'] > 0.6 else "🤔"
    gender_emoji = "👩" if result['gender'] == 'Female' else "👨"
    
    print(f"{gender_emoji} {name:10} → {result['gender']:6} ({result['confidence']*100:5.1f}% confident) {confidence_emoji}")

print("\n✅ Model is working correctly!")

In [ ]:
# 🌐 Flask API Server Setup
app = Flask(__name__)
CORS(app)

@app.route('/predict', methods=['POST'])
def predict():
    try:
        data = request.get_json()
        name = data.get('name', '')
        
        if not name:
            return jsonify({'error': 'No name provided'}), 400
        
        result = gender_model.predict(name)
        
        # Add training metrics
        if gender_model.training_history:
            latest_epoch = len(gender_model.training_history['accuracy']) - 1
            result['training_metrics'] = {
                'accuracy': gender_model.training_history['accuracy'][latest_epoch],
                'loss': gender_model.training_history['loss'][latest_epoch],
                'val_accuracy': gender_model.training_history['val_accuracy'][latest_epoch],
                'val_loss': gender_model.training_history['val_loss'][latest_epoch]
            }
        
        return jsonify(result)
        
    except Exception as e:
        return jsonify({'error': str(e)}), 500

@app.route('/model_info', methods=['GET'])
def model_info():
    try:
        summary_list = []
        if gender_model.model:
            gender_model.model.summary(print_fn=lambda x: summary_list.append(x))
        
        return jsonify({
            'model_summary': '\n'.join(summary_list),
            'training_history': gender_model.training_history,
            'vocab_size': len(gender_model.char_to_idx) if gender_model.char_to_idx else 0
        })
    except Exception as e:
        return jsonify({'error': str(e)}), 500

@app.route('/health', methods=['GET'])
def health():
    return jsonify({
        'status': 'healthy', 
        'model_loaded': gender_model.model is not None,
        'message': '🧠 Real Neural Network is ready!'
    })

print("✅ Flask API routes configured!")

In [ ]:
# 🚀 Start the server and create public URL
def run_flask():
    app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)

# Start Flask in background thread
flask_thread = threading.Thread(target=run_flask)
flask_thread.daemon = True
flask_thread.start()

# Wait a moment for Flask to start
time.sleep(3)

# Create ngrok tunnel
print("🌐 Creating public URL with ngrok...")
public_url = ngrok.connect(5000)

print("\n" + "="*60)
print("🎉 SUCCESS! Your neural network is now live!")
print("="*60)
print(f"\n📡 Public URL: {public_url}")
print(f"\n📋 Copy this URL and paste it in your React app to connect!")
print("\n🔗 Available endpoints:")
print(f"   • POST {public_url}/predict - Make predictions")
print(f"   • GET  {public_url}/model_info - Get model information")
print(f"   • GET  {public_url}/health - Check server health")
print("\n⚡ The server will keep running until you stop this cell.")
print("\n🎯 Now go to your React app and connect using the URL above!")

In [ ]:
# 🔄 Keep the server running and show live requests
print("🔄 Server is running... Watching for incoming requests:")
print("💡 Tip: Change names in your React app to see predictions here!")
print("-" * 60)

try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\n🛑 Server stopped by user")
    ngrok.disconnect(public_url)
    print("✅ ngrok tunnel closed")